# 03 — Análise Exploratória dos Dados (EDA)

Objetivo: compreender a camada Silver, avaliar qualidade, identificar vazamentos e definir as transformações da camada Gold para o modelo adaptativo.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

silver_path = Path("../data/silver/bank_marketing_silver.csv")
df = pd.read_csv(silver_path, sep=";")

print(f"Shape: {df.shape}")
df.head()


## 1. Visão geral e tipos

O dataset Bank Marketing possui ~41 mil registros com variáveis demográficas, histórico de campanha, indicadores econômicos e o target `y` (aceitação do depósito a prazo).


In [ ]:
df.info()


In [ ]:
df.describe(include="number").T


In [ ]:
df.describe(include=["object", "string"]).T


## 2. Balanceamento do target

Há forte desbalanceamento: a maioria dos clientes não converteu. Isso deve orientar a avaliação (não usar só acurácia) e a definição de recompensa no bandit.


In [ ]:
target_counts = df["y"].value_counts()
target_pct = df["y"].value_counts(normalize=True)

display(pd.DataFrame({"count": target_counts, "pct": target_pct.round(4)}))

ax = target_counts.plot(kind="bar", color=["#4C78A8", "#F58518"], figsize=(5, 3))
ax.set_title("Distribuição do target (y)")
ax.set_xlabel("y")
ax.set_ylabel("Frequência")
plt.tight_layout()
plt.show()


## 3. Distribuições categóricas relevantes


In [ ]:
cat_cols = ["job", "marital", "education", "housing", "loan", "contact", "month", "poutcome"]

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
axes = axes.ravel()

for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color="#4C78A8")
    ax.set_title(col)
    ax.set_xlabel("count")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


## 4. Taxa de conversão por categoria

Hipótese operacional: canal de contato, resultado de campanhas anteriores e mês influenciam a conversão — candidatos naturais a braços ou contexto no bandit.


In [ ]:
def conversion_by(col, top_n=None):
    tmp = df.copy()
    tmp["converted"] = (tmp["y"].astype(str).str.lower() == "yes").astype(int)
    out = (
        tmp.groupby(col, dropna=False)
        .agg(n=("converted", "size"), conversion=("converted", "mean"))
        .sort_values("conversion", ascending=False)
    )
    if top_n:
        out = out.head(top_n)
    return out

for col in ["contact", "poutcome", "month", "education", "job"]:
    print(f"\n=== Conversão por {col} ===")
    display(conversion_by(col).round(4))


## 5. Variáveis numéricas e correlação

`duration` correlaciona fortemente com conversão, mas só é conhecida após a chamada — **data leakage**. Não deve entrar no modelo preditivo/adaptativo.


In [ ]:
num_cols = [
    "age", "duration", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"
]
num_cols = [c for c in num_cols if c in df.columns]

corr_df = df[num_cols].copy()
corr_df["y_bin"] = (df["y"].astype(str).str.lower() == "yes").astype(int)

corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlação (numéricas + target binário)")
plt.tight_layout()
plt.show()

display(corr["y_bin"].sort_values(ascending=False).to_frame("corr_with_y"))


## 6. Outliers

Campanhas com muitos contatos (`campaign` alto) e `previous` extremo podem distorcer regras. Na Gold, `campaign` será agrupada em buckets.


In [ ]:
outlier_cols = [c for c in ["age", "duration", "campaign", "previous"] if c in df.columns]

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(4 * len(outlier_cols), 4))
if len(outlier_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color="#4C78A8")
    ax.set_title(col)

plt.tight_layout()
plt.show()

summary = []
for col in outlier_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    summary.append({"column": col, "q1": q1, "q3": q3, "iqr": iqr, "n_outliers_iqr": int(n_out)})

pd.DataFrame(summary)


## 7. Data leakage e qualidade

- **Remover `duration`**: conhecida só depois do contato; vaza o resultado da chamada.
- **`default`**: categoria extremamente desbalanceada / pouco informativa na prática — candidata a remoção na Gold.
- **`pdays == 999`**: convenção do dataset para "nunca contactado" → feature `never_contacted`.


In [ ]:
print("Valores ausentes:")
display(df.isna().sum().sort_values(ascending=False).to_frame("n_missing"))

print("\nDistribuição de default:")
display(df["default"].value_counts(dropna=False) if "default" in df.columns else "coluna ausente")

print("\npdays == 999 (nunca contactado):", int((df["pdays"] == 999).sum()))
print("Inconsistência previous==0 & poutcome==success:",
      int(((df["previous"] == 0) & (df["poutcome"].astype(str).str.lower() == "success")).sum()))


## 8. Hipóteses → transformações Gold

| Hipótese | Ação na Gold |
|---|---|
| `duration` vaza informação da chamada | Remover |
| `default` pouco útil / ruidoso | Remover |
| `pdays == 999` indica sem contato prévio | Criar `never_contacted` |
| Sucesso em campanha anterior aumenta conversão | Criar `previous_success` |
| Muitos contatos na campanha atual saturam | Criar `campaign_bucket` |
| Canal (`contact`) diferencia conversão | Manter como feature / braço candidato |

Próximo passo: executar `04_ingestion_gold.ipynb`.
